# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. It follows the Croissant schema and references all dataset entities by their `@id`.

### Dataset Source
The dataset schema is described in Croissant JSON-LD, accessible via the following URL:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset using the mlcroissant Dataset object
dataset = mlc.Dataset(croissant_url)

# Access metadata (as an object)
metadata = dataset.metadata
print("Dataset Name:", metadata.name)
print("Description:", metadata.description)
print("Identifier:", getattr(metadata, 'identifier', None))
print("Version:", getattr(metadata, 'version', None))

## 2. Data Overview
Let's review the available RecordSets, Fields, and their `@id`s.

We'll access each RecordSet and list its fields, columns, and their unique `@id`s. (The record set object can be found in `metadata.recordSets`, and each field is referenced by its `@id`.)

In [ ]:
# Print all available RecordSets and Fields with their @id values
print("Available record sets in this dataset:")
record_set_objs = getattr(metadata, 'recordSets', [])
if not record_set_objs:
    print("No record sets found in metadata (metadata.recordSets is empty).")
else:
    for rs in record_set_objs:
        print(f"- RecordSet name: {getattr(rs, 'name', None)}; @id: {getattr(rs, '@id', None)}")
        print("  Fields:")
        fields = getattr(rs, 'fields', [])
        for f in fields:
            print(f"    - Field name: {getattr(f, 'name', None)}, @id: {getattr(f, '@id', None)}, dataType: {getattr(f, 'dataType', None)}")
        columns = getattr(rs, 'columns', [])
        if columns:
            print("  Columns:")
            for c in columns:
                print(f"    - Column name: {getattr(c, 'name', None)}, @id: {getattr(c, '@id', None)}, dataType: {getattr(c, 'dataType', None)}")
        print()
# As a shortcut, collect all record set @ids
record_set_ids = []
for rs in getattr(metadata, 'recordSets', []):
    rsid = getattr(rs, '@id', None)
    if rsid:
        record_set_ids.append(rsid)
# Print record set IDs for reference
print("RecordSet IDs:")
pprint.pprint(record_set_ids)

## 3. Data Extraction
Load tabular data from each record set into a Pandas DataFrame for analysis. All references below use the actual record set and field `@id`s as shown above.

If no explicit record sets were defined in the schema, try to extract from the dataset default record set (e.g., the first one returned by `dataset.records()`).

In [ ]:
# Try to extract data from each available record set (by @id)
dataframes = {}
if record_set_ids:
    for rsid in record_set_ids:
        print(f"\nLoading records for record set @id: {rsid}")
        recs = list(dataset.records(record_set=rsid))
        df = pd.DataFrame(recs)
        dataframes[rsid] = df
        print(f"Loaded {len(df)} rows, columns: {df.columns.tolist()}")
    first_rsid = record_set_ids[0]
else:
    # If no record set is defined, try loading from default
    print("No record sets explicitly defined; loading records from default dataset...")
    recs = list(dataset.records())
    df = pd.DataFrame(recs)
    dataframes['default'] = df
    first_rsid = 'default'
    print(f"Loaded {len(df)} rows, columns: {df.columns.tolist()}")
# Display a sample of the main DataFrame
print(f"\nColumns in the first DataFrame ({first_rsid}):")
print(dataframes[first_rsid].columns.tolist())
dataframes[first_rsid].head()

## 4. Exploratory Data Analysis (EDA)
Now let's explore the contents of the main DataFrame. We'll process a numeric field (e.g., age, if available), filter by value, normalize, and group by a categorical field (e.g., sex or MSI status).

Please refer to the `@id` values for each field/column, as printed above, to select target columns for processing.

In [ ]:
df = dataframes[first_rsid]
print("Available columns in the selected DataFrame:")
print(df.columns.tolist())

# Choose a numeric field for demonstration (set to your column's @id)
# Example guesses (adjust according to actual dataset columns):
possible_numeric_fields = [col for col in df.columns if (('age' in col.lower()) or ('interval' in col.lower()) or (df[col].dtype in [int, float]))]

if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
    print(f"Selected numeric field: {numeric_field_id}")
    # Filter for records where value > 40 (example threshold)
    threshold = 40
    filtered_df = df[df[numeric_field_id].astype(float) > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    norm_col = numeric_field_id + '_normalized'
    filtered_df[norm_col] = (filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()) / filtered_df[numeric_field_id].astype(float).std()
    print(f"\nNormalized {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by a key categorical field (e.g., MSI status)
    possible_group_fields = [col for col in df.columns if ('sex' in col.lower() or 'msi' in col.lower() or df[col].dtype == object)]
    if possible_group_fields:
        group_field_id = possible_group_fields[0]
        print(f"\nGrouping by: {group_field_id}")
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(grouped.head())
else:
    print('No numeric columns found for EDA demonstration.')

## 5. Visualization
Let's visualize the distribution of the selected numeric field (if available), as well as relationships with a categorical variable, using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if possible_numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].astype(float), bins=20, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.show()

    if possible_group_fields:
        plt.figure(figsize=(8,4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.show()
else:
    print('No numeric field available for visualization.')

## 6. Conclusion

In this notebook, we demonstrated how to load, inspect, and analyze a medical research dataset in Croissant format using the `mlcroissant` library:

- Loaded dataset metadata and reviewed available record sets and schema fields by unique `@id`.
- Extracted records into dataframes for tabular analysis.
- Performed simple exploratory data analysis, filtering and normalizing a numeric field, and grouped by a clinical attribute.
- Visualized distributions and associations using matplotlib and seaborn.

The FAIR^2 dataset enables characterization of clinicopathological predictors in colorectal cancer survivors and illustrates the power of interoperable scientific data using standardized schemas.